# Integrasjon av diskrete verdier ved inndeling i celler

Noen ganger ønsker vi å integrere diskrete data for å finne et gjennomsnitt av data over et område. Forskjellen fra det vi har gjort tidligere er at vi ikke har en funksjon vi kan evaluere i vilkårlige nettverks-punkter. I stedet får vi oppgitt noen diskrete datapunkter og blir bedt om å beregne et gjennomsnitt over disse.

Dette betyr i praksis at vi må beregne en endelig Riemann-sum. Dette brukes som et dobbeltintegral. I motsetning til situasjonen tidligere kan vi ikke (uten problemer eller å undersøke problemet nærmere) gå til grensen slik vi ville gjort i funksjonstilfellet.
Likevel kan vi se dette som en forberedelse på nettbaserte metoder som vi senere skal bruke i kapitlet om partielle differensiallikninger.

Vi har sett at gjennomsnittsverdi for en funksjon $f(x,y)$ er gitt som

$$\overline{f} = \frac{1}{\text{arealet til }D} \iint_D f(x,y) dA$$

I tidligere notater har vi sett på hvordan vi kan diskretisere funksjoner ved hjelp av Numpys mesh funksjon. 
Problemet vi skal nå kort se på hvordan vi kan håndtere diskret data. For eksempel hvis $f(x,y)=k$ er konstant, så gjelder 

$$\overline{f} = \frac{1}{\text{arealet til }D} \iint_D \underbrace{f(x,y)}_{=k} dA = \frac{1}{\text{arealet til }D} \cdot k \cdot\text{arealet til }D =k$$

Mer generelt, hvis vi kan dele $D$ i områder $D=D_1\cup D_2 \cup \ldots \cup D_n, n\in \mathbb{N}$ slik at $f(x,y)=k_i$ er konstant på hver $D_i$ så gjelder
$$\iint_D f(x,y) dA = \iint_{D_1}\underbrace{f(x,y)}dA + \cdots + \iint_{D_n} f(x,y)dA =  k_1 \cdot \text{areal til } D_1 + \cdots k_n \cdot \text{areal til } D_n    $$
Dermed er for en funksjon som bare ta endelig mange verdier på $D$:
$$ \overline{f} = \frac{1}{\text{arealet til }D} \iint_D f(x,y) dA = \sum_i k_i \frac{\text{areal til }D_i}{\text{areal til }D}$$

In [147]:
import numpy as np

## Eksempel: Klimadata over Norge

Et vanlig eksempel hvor vi ser på diskrete data er vær- og klimadata. Disse samles inn ved hjelp av målinger i noen steder i landet. Her ser vi på et eksempel nedbør kart fra ekstremvær "Hans" i Norge (kilde Wikipedia)

<img src="https://upload.wikimedia.org/wikipedia/commons/8/8f/Nedb%C3%B8r_mandag_7._august_2023.png" alt="Nedbør mandag 7. august 2023" width="800"/>


Kart viser gjennomsnittlig nedbør over Sør-Norge. Kartet viser Vi kan lure nå om gjennomsnittlig nedbør i hele Sør-Norge på 07.08.2023 ifølge ekstremvær Hans. For dette må vi lage et rutenett over kartet.
Det gjør vi slik: 
Plasser origo i det sørvestlige hjørnet av kartet slik at alle verdier kan betraktes som liggende i første kvadrant, og dermed er alle positive. Del deretter hele kartet inn i rektangler. Vi bruker har en oppdeling som ble lagd av satellitdata fra Copernikus Sentinel 2 satellit som vist nede. 

<img src="https://ec.europa.eu/eurostat/statistics-explained/images/c/cd/2018_NO_01_GG2023.jpg" alt="Sentinel 2 satellitdata, indeling Norge" width="800"/>

Cellene i bildet over er 100km x 100km stor. Vi ignorerer for nå at de ligger over hverandre ved rand (vi lagger en tilnærming uansett. (OBS: Verden er ikke en kvadratisk overflate så det er ikke forundrelig at det er ikke mulig å plassere kvadrater uten overlapp pent på en kart av Norge). For å lage en tilnærming kan vi bruke de 16 celler som ligger i den 4x4 blokk dens sør-vest- hjørnet ligger fra andre cellen til venstre av den røde cellen. Hvis vi estimerer fra nedbør kartet gjennomsnittlig nedbør i celler får vi en matrise $ij$-indeks i matrisen tilsvarer plass til cellene i den 4x4 gitter. 

In [148]:
Kmatrise = np.array([[40, 40, 85, 65],[50, 75, 90, 20], [37, 52, 150, 60], [22,48,88,70]])
Kmatrise

array([[ 40,  40,  85,  65],
       [ 50,  75,  90,  20],
       [ 37,  52, 150,  60],
       [ 22,  48,  88,  70]])

Siden alle celler er like stor (her akkurat $\frac{1}{16}$), får vi fra formelen for middelverdien at 
$$\text{gjennomsnittlig regnmengde} = \frac{1}{16}\sum_{i,j} Kmatrise_{ij}

In [149]:
Kmatrise.size

16

In [150]:
gjen_regn = np.sum(Kmatrise)/Kmatrise.size


# Calculate

print('Gjennomsnittlig regnmengde i Sør-Norge ved ekstremvær Hans var ', gjen_regn, 'mm')

Gjennomsnittlig regnmengde i Sør-Norge ved ekstremvær Hans var  62.0 mm


## Regn på kysten

Under ekstremvær ble også regnmengden ved kysten målt. Hvis vi ser på celler som ligger vest til de celler vi har sett på finner vi celler ved kysten.
Regnmålinger for disse celler sammler vi i listen:

In [151]:
regn_kysten = np.array([12,35,20,53])

Vi kan enkelt utvide gjennomsnittlig regnmengde ved å bruke de celler i den nye listen i tillegg til det vi har.

$$
\text{regn innlandet og kysten}
= \frac{1}{20}\left(
\underbrace{\sum_{i,j} \text{Kmatrise}_{ij}}_{\text{bidrag innlandet}}
+ \underbrace{\sum_k \text{regn-kysten}_k}_{\text{kystens bidrag}}
\right)
$$

Merk at vi trenger å oppdatere tallet av celler vi deler med siden vi nå har $20$ celler.


In [152]:
gjen_regn_med_kyst = (np.sum(Kmatrise) + np.sum(regn_kysten))/(Kmatrise.size + regn_kysten.size)

## Calculate

print('Gjennomsnittlig regnmengde i Sør-Norge og kysten ved ekstremvær Hans var ', gjen_regn_med_kyst, 'mm')

Gjennomsnittlig regnmengde i Sør-Norge og kysten ved ekstremvær Hans var  55.6 mm


De nye celler vi har tilføyet er ved kysten og inneholder dermed en god del hav hvor vi har ingen målinger siden målestasjoner ligger på land. 
Det er dermed ikke ønskelig å behandle disse celler som alle andre celler vi har allerede sett på siden de inneholder så mye hav.

### Ide: Reduser andel av nye celler i resultatet!

Når vi ser på kart er det maks 15% andel av land i enhver av cellene ved kysten. Dermed skulle bidrag til resultatet i disse cellene ikke overstige 15% av en full celle. 
Vi kan implementere det ved hjelp av å definere vekt til cellene:

Så hvis cellene ved kysten vektes bare som 15% av en full land celle kan vi multiplisere verdien av cellen med $0.15$ for å korrigiere verdien i summen.

OBS: Nå har vi 16 fulle celler og 4 celler som vektes hver med 15%. Dermed får vi ulike vekt for begge deler av summen. Se på følgende kode:

In [153]:
gjen_regn_korrigert_kyst = (np.sum(Kmatrise) + np.sum(regn_kysten)*0.15  )/(Kmatrise.size + regn_kysten.size*0.15)

## Calculate

print('Gjennomsnittlig regnmengde i Sør-Norge og kysten ved ekstremvær Hans var ', gjen_regn_korrigert_kyst, 'mm')

Gjennomsnittlig regnmengde i Sør-Norge og kysten ved ekstremvær Hans var  60.843373493975896 mm


## Sammenlikning av resultater
Tabellen nede viser sammenlikning av resultater.

In [154]:
from tabulate import tabulate

# Data to display
headers = ["Beregning for ", "Gjennomsnittlig regn (mm)"]
data = [
    ["Innlandet uten kysten", gjen_regn],
    ["Innlandet og kysten", gjen_regn_med_kyst],
    ["Innlandet og kyst korrigert", gjen_regn_korrigert_kyst]
]

# Display the table
print(tabulate(data, headers=headers, tablefmt="grid"))

+-----------------------------+-----------------------------+
| Beregning for               |   Gjennomsnittlig regn (mm) |
+=============================+=============================+
| Innlandet uten kysten       |                     62      |
+-----------------------------+-----------------------------+
| Innlandet og kysten         |                     55.6    |
+-----------------------------+-----------------------------+
| Innlandet og kyst korrigert |                     60.8434 |
+-----------------------------+-----------------------------+


## Hvordan kan vi implementere det mer konsepsjonelt?

VI har endret hvor mye en celle bidrar i siste beregning men har bare skillet to tilfeller:
1. Cellen er i innlandet
2. Cellen er ved kysten

Det kan godt være at vi ønsker å vekte cellene ulikt fordi di bidra ulik og vi har for hver cellen sin egen vekt som implementerer det.
I dette tilfelle blir det fort tungvint å implementere beregning på den måten vi har gjort:

$$\frac{1}{20}\left(\underbrace{\sum_{i,j} \text{Kmatrise}_{ij}}_{\text{bidrag  innlandet}} + \underbrace{\sum_k \text{regn-kysten}_k}_{\text{kystens bidrag}}\right)$$

Det vi egentlig vil er å beregne for hver celle

$$\text{verdi i cellen} \cdot \text{ vekt av cellen}$$

oppsummerer disse og dele med summen av vektene. Formelen blir der

$$\text{gjennomsnittlig regnmengde} = \frac{1}{\text{summen av vekter}}\sum_{celler} \text{verdi i cellen} \cdot \text{vekt av cellen}$$


In [155]:
vekter_kysten = np.array([.15, .15, .15,.15])
vekter_innlandet = np.ones((4,4)) 

In [156]:
gjen_regn = (np.sum(Kmatrise*vekter_innlandet) + np.sum(regn_kysten * vekter_kysten) ) / (np.sum(vekter_innlandet) + np.sum(vekter_kysten))
## Calculate

print(gjen_regn)

60.843373493975896


Resultatet er det samme som tidligere, men vi kan nå enkelt endre noen av de vektene for å endre hvordan en celle bidrar (uten at vi må tilpasse noe annet i koden).

For eksempel, hvis den siste kystcellen bidrar enda mindre (fordi landmasse i cellen er mindre enn 5%), så kan vi enkelt endre verdien til vekt for å beregne gjennomsnittsverdien på nytt.

In [157]:
vekter_kysten[3]=.05
print(vekter_kysten)

[0.15 0.15 0.15 0.05]


In [158]:
regn_ny = (np.sum(Kmatrise*vekter_innlandet) + np.sum(regn_kysten * vekter_kysten) ) / (np.sum(vekter_innlandet) + np.sum(vekter_kysten))

## Calculate

print('Gjennomsnittlig regnmengde ved ekstremvær Hans, nye vekt, var ', regn_ny, 'mm')

Gjennomsnittlig regnmengde ved ekstremvær Hans, nye vekt, var  60.89090909090909 mm


# Lekse: Coderunner

Målsetning med oppgaven er å ta være på overlapp mellom de cellene vi har sett på i problemet om nedbørmengden. Vi repeterer bildet igjen og peker på at overlapp fører til at cellene bidrar faktisk dobbelt til regnmengden over noen områder:

<img src="https://ec.europa.eu/eurostat/statistics-explained/images/c/cd/2018_NO_01_GG2023.jpg" alt="Sentinel 2 satellitdata, indeling Norge" width="400"/>

Ideen er nå at vi skal tilpasse koden slik at vi ungår dobbeltvurdering av overlappen.

## Forenklet eksempel: Overlapp bare i vest/øst retning

Vi starter med overlapp i vest/øst retning. La oss se på dett ved hjelp av et grafisk eksempel som representerer en råd i cellene vi skal se på.

![Råd med overlapp](https://raw.githubusercontent.com/imfdrift/IMAx3012/main/Ressources/squares1.png)

Her vises overlapp ved hjelp av blå farge.

Som vi kan se er overlapp mellom naboceller dobbelt vurdert, både med verdien fra cellen til venstre og cellen fra høyre. For å unngå dobbeltelling av overlapp gir det mening å behandle hver overlapp som sin egen celle. Så det vi skal gjøre er å gi disse nye celler en verdi og deretter fordeler vi nye vekter for både de nye celler og de gamle vi har.

### Hvordan finner vi verdien til overlapp mellom to celler?

La oss si, vi skal se på overlapp av celle 1 og 2, da er det hensiktsmessig å anta at verdien for denne cellen er gjennomsnitt av verdien i celle 1 og verdien i celle 2:

$$\text{Verdi overlapp} = \frac{\text{verdi celle 1} + \text{verdi celle 2}}{2}$$

Så når vi vet verdien til cellene kan vi enkelt beregne verdien til de nye celler.
La oss gjøre det med eksempel råd fra forelesning hvor tallene representerer gjennomsnittlig nedbør. 

In [159]:
Nedboer_rad1 = np.array([40, 40, 85, 65])

#Calculate

overlapp_rad1 = (Nedboer_rad1[:-1] + Nedboer_rad1[1:]) / 2

print("Verdier i råd:", Nedboer_rad1)
print("Gjennomsnitt for overlapp:", overlapp_rad1)

Verdier i råd: [40 40 85 65]
Gjennomsnitt for overlapp: [40.  62.5 75. ]


## Vekter for cellene 

Vi må nå bestemme vekter for cellene. For dette må vi vite hvor stor overlappen er. Vi skal anta at overlapp mellom to celler samsvarer $7.5%$ av hele cellen.

Dermed vet vi at vektene bør være:
1. $.925$ for en celle som overlapper med en annen celle
2. $.85$ for en celle som overlapper med to andre celler
3. $.075$ for overlappen mellom to celler

Vi kan lage det nå som to matriser:

In [160]:
## Calculate

vekt_orig_celler = np.array([0.925, 0.85, 0.85, 0.925])
vekt_overlapp    = np.array([0.075, 0.075, 0.057])

# Vi beregner gjennomsnittlig regnmengde nå som

sum_vekter = np.sum(vekt_orig_celler) + np.sum(vekt_overlapp)
regnmengde = (np.sum(vekt_orig_celler*Nedboer_rad1) + np.sum(vekt_overlapp * overlapp_rad1))/sum_vekter

print("Gjennomsnittlig regnmengde i råd uten dobbeltelling er:", regnmengde, "mm")

Gjennomsnittlig regnmengde i råd uten dobbeltelling er: 57.31634282672346 mm


Formelen i siste cellen beregner 

$$\text{regnmengde} = \frac{\sum_{\text{celler}} \text{verdi i cellen} \cdot \text{vekt til cellen} + \sum_{\text{overlapp}} \text{verdi i overlapp} \cdot \text{vekt til overlapp}}{\text{summe av vekter}}$$

Problemet nå er at det tar bare være på overlapp i vest øst retning. Som er synlig på kart har vi både overlapp i vest-øst og i nord-sør retning. Det er nå deres oppgave å ta være på det i oppgaven.

## 

## Overlapp nå i 2D

Vi ser (igjen) på en forenklet modell og anta at vi har bare to råd av celler

![To råd med overlapp](https://raw.githubusercontent.com/imfdrift/IMAx3012/main/Ressources/squares2.png)

Nå finnes det to typer overlapp: I vest-øst retning (igjen i blå) og i nord-sør retning (grønn).
Vi kopierer igjen verdiene til regnmende, men må beregne nå to typer overlapp:

1. Overlapp i vest-øst retning for hver råd (dvs. vi får 2x 3 celler)
2. Overlapp i nord-sør retning mellom rådene (dvs. vi får 1 x 4 celler)

#### Oppgave: Lag kode som lager to matriser med verdier til overlappceller. Veriden skal igjen være  gjennomsnittsverdi av de to naboceller som møter hverandre. 

In [161]:
to_rad_matrise = np.array([[40, 40, 85, 65],[50, 75, 90, 20]])

# Calculate

#Vi programmerer direkte slik at 
overlapp_horisontal = (to_rad_matrise[:,:-1] + to_rad_matrise[:,1:])/2
print(overlapp_horisontal)

overlapp_vertikal = (to_rad_matrise[:-1,:] + to_rad_matrise[1:,:])/2
print(overlapp_vertikal)


[[40.  62.5 75. ]
 [62.5 82.5 55. ]]
[[45.  57.5 87.5 42.5]]


## Vekter for de celler

Vi skal anta igjen at overlapp (både i vest-øst og i nord-sør retning) er $7.5\%$ av en hel celle.
Dermed har vi nå to forskjellige typer celler som får ulike vekt:

1. Celler i hjørnene har 2 overlapp og dermed får vekt $1-2\cdot 0.075=0.85$
2. Celler med tre overlapp får vekt $1-3\cdot 0.075=0.775$
3. Alle celler i overlapp får vekt $0.075$

### Oppgave:
1. Lag tre matriser som inneholder riktig vekt for alle cellene (Hint: Det kunne være lurt å bruke en tegning)
2. Beregn gjennomsnittlig regn uten å telle overlapp mellom to celler dobbelt.

In [163]:
# Calculate

vekt_orig_cell = np.array([[0.85, 0.775, 0.775, 0.85],[0.85, 0.775, 0.775, 0.85]])
vekt_hor_overlapp = np.array([[0.075, 0.075,0.075],[0.075, 0.075,0.075]])
vekt_vert_overlapp = np.array([0.075, 0.075, 0.075, 0.075])

#Vi regner nå som før

sum_vekt = np.sum(vekt_orig_cell) + np.sum(vekt_hor_overlapp) + np.sum(vekt_vert_overlapp)
print(sum_vekt)
gjennomsnittregn_to_rad = (np.sum(vekt_orig_cell * to_rad_matrise) + np.sum(overlapp_horisontal* vekt_hor_overlapp) + np.sum(overlapp_vertikal * vekt_vert_overlapp))/sum_vekt

print("Gjennomsnittlig regnmengde i råd uten dobbeltelling er:", gjennomsnittregn_to_rad, "mm")

7.25
Gjennomsnittlig regnmengde i råd uten dobbeltelling er: 57.827586206896555 mm
